In [1]:
# ============================================================
# NB10_Regime_Allocator (FIXED)
# Dynamically allocates advisory portfolio weights by market regime
# Output: gold_portfolio_allocation
# ============================================================

import uuid
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
TODAY = datetime.utcnow().date()

# ------------------------------------------------------------
# 1. Get Latest Market Regime (NameError-safe)
# ------------------------------------------------------------
regime_df = (
    spark.table("silver_market_regime")
    .orderBy(F.col("trade_date").desc())
    .limit(1)
    .select("market_regime", "trade_date", "close", "sma_200")
    .toPandas()
)

# Initialize defaults BEFORE the conditional
regime_status = "UNKNOWN"
nifty_close = None
nifty_sma = None
pct_from_sma = 0.0

if len(regime_df) == 0:
    print("No market regime data found. Defaulting to UNKNOWN.")
else:
    row = regime_df.iloc[0]
    regime_status = str(row["market_regime"])
    nifty_close = float(row["close"]) if pd.notna(row["close"]) else None
    nifty_sma = float(row["sma_200"]) if pd.notna(row["sma_200"]) else None

    if nifty_close is not None and nifty_sma is not None and nifty_sma > 0:
        pct_from_sma = ((nifty_close / nifty_sma) - 1.0) * 100.0
        print(f"Market Regime : {regime_status}")
        print(f"Nifty Close   : {nifty_close:,.2f}")
        print(f"Nifty SMA200  : {nifty_sma:,.2f}")
        print(f"Distance      : {pct_from_sma:+.2f}% from SMA200")
    else:
        print(f"Market Regime : {regime_status} (price data incomplete)")

# ------------------------------------------------------------
# 2. Define Allocation Rules by Regime
# ------------------------------------------------------------
if regime_status == "RISK_ON":
    allocation = {
        "GRAHAM":     0.50,
        "GREENBLATT": 0.20,
        "ONEIL":      0.30,
        "LYNCH":      0.00,
        "BUFFETT":    0.00,
        "CASH":       0.00,
    }
    regime_label = "Bull Market - Growth and momentum permitted"

elif regime_status == "RISK_OFF":
    allocation = {
        "GRAHAM":     0.70,
        "GREENBLATT": 0.20,
        "ONEIL":      0.00,
        "LYNCH":      0.00,
        "BUFFETT":    0.00,
        "CASH":       0.10,
    }
    regime_label = "Bear Market - Defensive value only"

else:
    allocation = {
        "GRAHAM":     0.60,
        "GREENBLATT": 0.20,
        "ONEIL":      0.10,
        "LYNCH":      0.00,
        "BUFFETT":    0.00,
        "CASH":       0.10,
    }
    regime_label = "Uncertain regime - Balanced allocation"

# ------------------------------------------------------------
# 3. Load Latest Candidates
# ------------------------------------------------------------
try:
    candidates = spark.table("gold_stock_candidates").toPandas()

    # Filter to latest screen_date if the column exists
    if len(candidates) > 0 and "screen_date" in candidates.columns:
        latest_date = candidates["screen_date"].max()
        candidates = candidates[candidates["screen_date"] == latest_date]

    total_candidates = len(candidates)
except Exception as e:
    print(f"Could not load candidates: {e}")
    candidates = pd.DataFrame()
    total_candidates = 0

# Count by research_status tag
if total_candidates > 0 and "research_status" in candidates.columns:
    status_counts = candidates["research_status"].fillna("UNTAGGED").value_counts().to_dict()
else:
    status_counts = {}

print(f"\nCandidates loaded: {total_candidates}")
if status_counts:
    print("Research status breakdown:")
    for status, count in sorted(status_counts.items()):
        print(f"  {status:30s} : {count}")

# ------------------------------------------------------------
# 4. Map Styles to Actual research_status Tags from NB06
# ------------------------------------------------------------
STATUS_MAP = {
    "GRAHAM":     ["CORE_VALUE"],
    "GREENBLATT": ["CASH_COMPOUNDER", "CORE_CANDIDATE"],
    "ONEIL":      ["GROWTH_CONFIRMED", "SEASONAL_GROWTH_STREAK"],
    "LYNCH":      [],
    "BUFFETT":    [],
    "CASH":       [],
}

alloc_rows = []

for style, weight in allocation.items():
    tags = STATUS_MAP.get(style, [])
    matched_picks = sum(status_counts.get(tag, 0) for tag in tags)

    alloc_rows.append({
        "style": style,
        "target_weight": float(weight),
        "target_weight_pct": round(weight * 100, 1),
        "matched_candidates": int(matched_picks),
        "total_candidates": int(total_candidates),
        "weight_per_pick": round(weight / matched_picks, 4) if matched_picks > 0 else 0.0,
        "regime": regime_status,
        "regime_label": regime_label,
        "nifty_close": nifty_close,
        "nifty_sma_200": nifty_sma,
        "nifty_vs_sma200_pct": round(pct_from_sma, 2),
        "allocation_date": TODAY,
        "run_timestamp": RUN_TIMESTAMP,
        "run_id": RUN_ID
    })

alloc_df = pd.DataFrame(alloc_rows)

# ------------------------------------------------------------
# 5. Save to Lakehouse
# ------------------------------------------------------------
(
    spark.createDataFrame(alloc_df)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_portfolio_allocation")
)

print(f"\nSaved gold_portfolio_allocation ({len(alloc_df)} styles)")
print(f"Regime: {regime_status} | Label: {regime_label}")

display(alloc_df)

StatementMeta(, 23171979-c517-4331-bdf2-5235f6fcfcc4, 3, Finished, Available, Finished, False)

Market Regime : RISK_OFF
Nifty Close   : 22,603.05
Nifty SMA200  : 24,306.92
Distance      : -7.01% from SMA200

Candidates loaded: 15
Research status breakdown:
  CASH_COMPOUNDER                : 1
  CORE_CANDIDATE                 : 9
  CORE_VALUE                     : 3
  GROWTH_ACCELERATING            : 2

Saved gold_portfolio_allocation (6 styles)
Regime: RISK_OFF | Label: Bear Market - Defensive value only


SynapseWidget(Synapse.DataFrame, ed6aa899-9171-46f7-80d2-17adcd7c4255)